<a href="https://colab.research.google.com/github/AllenGeorge08/NeuralNetworksFromScratch/blob/master/CustomTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Trained on Custom Data

In [3]:
with open('input.txt','r') as f:
  raw_data = f.read()

print(f"Length of dataset : {len(raw_data)}")

Length of dataset : 8368


In [4]:
chars = sorted(list(set(raw_data)))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)


 !"'(),.0123456789:=?ABCDEFGHIKLMNOPRSTUWYZabcdefghijklmnopqrstuvwxyz
70


In [5]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(decode(encode("Hi my name is Allen")))

Hi my name is Allen


In [6]:
import torch
data = torch.tensor(encode(raw_data),dtype=torch.long) #64 bit integers
data.dtype,data.shape

(torch.int64, torch.Size([8368]))

In [7]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [24]:
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Batching (B,T)
batch_size=4 #four seq processed together
block_size=8 #each seq contains inputs (rows)

def get_batch(split):
  data = train_data if split=='train' else val_data
  ix = torch.randint(len(data)-block_size,(batch_size,))
  x = torch.stack([data[i:i + block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xtr,ytr  = get_batch('train')
# xtr.shape,ytr.shape
# ix.shape

torch.Size([4])

In [9]:
for b in range(batch_size):
  for t in range(block_size):
    context = xtr[b,:t+1]
    target = ytr[b,t]
    print(f"When input is {context} target is {target}")

When input is tensor([58]) target is 61
When input is tensor([58, 61]) target is 62
When input is tensor([58, 61, 62]) target is 1
When input is tensor([58, 61, 62,  1]) target is 63
When input is tensor([58, 61, 62,  1, 63]) target is 58
When input is tensor([58, 61, 62,  1, 63, 58]) target is 1
When input is tensor([58, 61, 62,  1, 63, 58,  1]) target is 56
When input is tensor([58, 61, 62,  1, 63, 58,  1, 56]) target is 48
When input is tensor([61]) target is 58
When input is tensor([61, 58]) target is 57
When input is tensor([61, 58, 57]) target is 48
When input is tensor([61, 58, 57, 48]) target is 0
When input is tensor([61, 58, 57, 48,  0]) target is 0
When input is tensor([61, 58, 57, 48,  0,  0]) target is 22
When input is tensor([61, 58, 57, 48,  0,  0, 22]) target is 1
When input is tensor([61, 58, 57, 48,  0,  0, 22,  1]) target is 47
When input is tensor([58]) target is 57
When input is tensor([58, 57]) target is 57
When input is tensor([58, 57, 57]) target is 48
When inpu

In [10]:
# Entire computation performed by open attention head
# Input x(B,T,C) -> Q(Query),K(Key),V(Value) -> QxK.t/math.sqrt Calculate attention scores: (B,T,T) -> Causal mask (Softmax) turn scores into weights -> weights x V. (Output: B,T,head_size)
# Each head uses 384//6 = 64 features
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 64
block_size = 256
max_iters = 500
eval_interval = 5000
learning_rate=3e-4
eval_iters=50

device='cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 384
n_head = 6
n_layer = 6
dropout=0.2
torch.manual_seed(1337)

device

'cuda'

In [26]:
# Single attention head
class Head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    B,T,C=x.shape
    k = self.key(x)
    q = self.query(x)
    # scale attention scores by 1/sqrt(head_size)
    wei = q @ k.transpose(-2,-1) * (k.size(-1)**-0.5)

    # Prevent looking at future positions
    wei = wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v
    return out

In [27]:
class MultiHeadAttention(nn.Module):
  def __init__(self,num_heads,head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd,n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.proj(out)
    out = self.dropout(out)
    return out

In [28]:
class FeedForward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
        nn.Dropout(dropout)
    )

  def forward(self,x):
    return self.net(x)

In [29]:
# Attention -> Feed-Forward-Network -> Wrapped with ressidual connections and preced bby LayerNorm
class TransformerBlock(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd//n_head
    self.sa = MultiHeadAttention(n_head,head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    x = x+self.sa(self.ln1(x))
    x = x+self.ffwd(self.ln2(x))
    return x


In [33]:
class LanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table = nn.Embedding(block_size,n_embd)
    self.blocks = nn.Sequential(*[TransformerBlock(n_embd,n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    B,T=idx.shape
    token_embedding = self.token_embedding_table(idx)
    position_embedding = self.position_embedding_table(torch.arange(T,device=idx.device))
    x = token_embedding+position_embedding
    x = self.blocks(x)
    x  = self.ln_f(x)
    logits = self.lm_head(x)
    if targets is None:
      loss  = None
    else:
      B,T,C=logits.shape
      logits = logits.reshape(B*T,C)
      targets = targets.reshape(B*T)
      loss = F.cross_entropy(logits,targets)
    return logits,loss

@torch.no_grad()
def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits,loss = self(idx_cond)
      # Generation only needs the final position
      if loss is None:
        logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs,num_samples=1)



In [34]:
model = LanguageModel(vocab_size)
model.to(device)
print(sum(p.numel() for p in model.parameters())/1e6,'M Parameters')
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)
for iter in range(max_iters):
  xtr,ytr = get_batch('train')
  xtr = xtr.to(device)
  ytr = ytr.to(device)
  logits,loss = model(xtr,ytr)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
  if iter%eval_interval ==0:
    print(f"Step: {iter}: train loss {loss.item():.4f}")

print("Final Loss : ",loss.item())

10.697542 M Parameters
Step: 0: train loss 4.2005
Final Loss :  2.2998194694519043
